# Email Requirement Discovery (unsupervised theme mining)

**Goal:** surface *what users are asking for* from ingested emails, when you don't yet
know the requirement space. Discovery, not classification — no labels, no fixed output
schema. The model proposes candidate themes; **you** decide what is a real requirement
vs. an artifact.

Pipeline: load -> clean -> filter noise -> embed -> cluster (BERTopic) -> inspect ->
land two tables in the **silver** lakehouse.

## 0. Install
On Fabric, `%pip install` restarts the PySpark kernel once (you will see the restart
warning). Run this cell on its own, first. Everything defined before the restart is wiped
— which is exactly why CONFIG comes *after* it. Then run the rest top-to-bottom.

In [ ]:
%pip install -q bertopic sentence-transformers umap-learn hdbscan
# %pip install -q talon   # optional: proper reply+signature extraction (install can be fussy)

## 1. CONFIG — the knobs you own
**This is the cell that was missing**, which is why everything downstream threw
`NameError`. It defines the column names the loader produces, the **silver** write
target, and the clustering knobs. Re-run it after any kernel restart.

In [ ]:
# --- source columns (must match what the loader produces) ---
ID_COL, SUBJECT_COL, TEXT_COL = "message_id", "subject", "body"

# --- SILVER destination (GUIDs taken from this notebook's lakehouse bindings) ---
# Sandbox-fine to hardcode; promote these to parameters if this becomes a pipeline.
WORKSPACE_ID  = "ac490e92-90f3-41a9-82ae-825ecaa77238"
SILVER_LH_ID  = "a03cfff1-048d-457c-8848-da958470832d"   # lh_silver_banking_data
SILVER_TABLES = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{SILVER_LH_ID}/Tables"
ASSIGN_TABLE  = "email_topic_assignments"
SUMMARY_TABLE = "email_topic_summary"
# Name-based equivalent (if you prefer readability over GUIDs):
# f"abfss://<workspace>@onelake.dfs.fabric.microsoft.com/lh_silver_banking_data.Lakehouse/Tables"

# --- boilerplate to strip. Synthetic business emails are short, so likely minimal —
#     inspect (cell 7) and add real patterns here only if a footer leaks into topics. ---
DISCLAIMER_PATTERNS = [
    r"This (e-?mail|message) and any attachments.*",
    r"CONFIDENTIALITY NOTICE.*",
]

# --- model / clustering (tuned for a small ~900-doc corpus) ---
EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # English; multilingual? paraphrase-multilingual-MiniLM-L12-v2
MIN_TOPIC_SIZE  = 10   # 20 is too coarse for <1k docs; smaller -> more, finer topics
MIN_TOKENS      = 5    # drop emails shorter than this after cleaning
RANDOM_STATE    = 42   # UMAP is stochastic; pin it for reproducibility

## 2. Load from Fabric Lakehouse Files
Reads `.eml` from the bronze lakehouse (the attached default), parses MIME, builds `pdf`
with message_id / subject / body. Confirmed working: 879 emails across 80 month-folders.
The `*.eml` glob is load-bearing — each email has a `.pdf` twin you do not want.

In [ ]:
from email import policy
from email.parser import BytesParser
import pandas as pd

try:
    fs = notebookutils.fs
except NameError:
    fs = mssparkutils.fs

BASE = "Files/bronze_raw/banking_data"
SCAN_ALL_MONTHS = True
ONE_MONTH = "2019/02"

def email_dirs():
    if not SCAN_ALL_MONTHS:
        return [f"{BASE}/{ONE_MONTH}/emails"]
    out = []
    for y in fs.ls(BASE):
        if not y.isDir: continue
        for m in fs.ls(y.path):
            p = f"{m.path}/emails"
            try: fs.ls(p); out.append(p)
            except Exception: pass
    return out

dirs = email_dirs()
print(f"{len(dirs)} email folder(s)")

bdf = (spark.read.format("binaryFile")
       .option("pathGlobFilter", "*.eml").load(dirs)
       .select("path", "content").toPandas())
rows = []
for _, r in bdf.iterrows():
    msg = BytesParser(policy=policy.default).parsebytes(r["content"])
    part = msg.get_body(preferencelist=("plain", "html"))
    rows.append({
        "message_id": r["path"].split("/")[-1],
        "subject": msg["subject"] or "",
        "body": part.get_content() if part else "",
    })
pdf = pd.DataFrame(rows)
print(f"{len(pdf):,} emails loaded")

## 3. Clean — strip quoted history, signatures, disclaimers
This is where quality is won or lost. The model never sees the raw body; it sees only
what survives this cell.

In [ ]:
import re

REPLY_MARKERS = [
    r"\nOn .{0,120}? wrote:",
    r"\n-{2,}\s*Original Message\s*-{2,}",
    r"\nFrom:\s.*?\nSent:\s.*?\nTo:\s",
    r"\n_{5,}",
    r"\nSent from my \w+",
]
SIGNOFF_RE = re.compile(
    r"\n\s*(kind regards|best regards|regards|thanks|thank you|cheers|sincerely)[,.]?\s*\n",
    re.IGNORECASE,
)
DISCLAIMER_RES = [re.compile(p, re.IGNORECASE | re.DOTALL) for p in DISCLAIMER_PATTERNS]

def clean_email(text: str) -> str:
    if not isinstance(text, str):
        return ""
    cut = len(text)
    for pat in REPLY_MARKERS:
        m = re.search(pat, text)
        if m:
            cut = min(cut, m.start())
    text = text[:cut]
    text = "\n".join(l for l in text.splitlines() if not l.lstrip().startswith(">"))
    for rx in DISCLAIMER_RES:
        text = rx.sub("", text)
    m = SIGNOFF_RE.search(text)
    if m:
        text = text[:m.start()]
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compose(row):
    subj = (row[SUBJECT_COL] + ". ") if SUBJECT_COL and isinstance(row.get(SUBJECT_COL), str) else ""
    return (subj + clean_email(row[TEXT_COL])).strip()

pdf["clean_text"] = pdf.apply(compose, axis=1)

## 4. Filter noise (auto-replies, OOO, empties)
Clustering does not know that "Out of Office" is not a requirement. Drop the obvious
non-signal before it forms its own confident, useless cluster.

In [ ]:
NOISE_RE = re.compile(
    r"\b(out of office|automatic reply|auto-reply|do not reply|delivery (has )?failed|"
    r"undeliverable|read receipt|unsubscribe)\b",
    re.IGNORECASE,
)

pdf["n_tokens"] = pdf["clean_text"].str.split().str.len()
mask = (pdf["n_tokens"] >= MIN_TOKENS) & (~pdf["clean_text"].str.contains(NOISE_RE))
dropped = (~mask).sum()
pdf = pdf[mask].reset_index(drop=True)
print(f"Dropped {dropped:,} noise/short emails -> {len(pdf):,} remain")

docs = pdf["clean_text"].tolist()

## 5. Embed

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBEDDING_MODEL)
embeddings = embedder.encode(docs, show_progress_bar=True, batch_size=64)

## 6. Cluster with BERTopic
UMAP -> HDBSCAN -> c-TF-IDF keywords. Topic count is *data-driven*. Topic `-1` is the
outlier bucket — on a small corpus expect it to be sizeable; that is HDBSCAN being honest,
not a bug.

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric="cosine", random_state=RANDOM_STATE)
hdbscan_model = HDBSCAN(min_cluster_size=MIN_TOPIC_SIZE, metric="euclidean",
                        cluster_selection_method="eom", prediction_data=True)

topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True,
)
topics, probs = topic_model.fit_transform(docs, embeddings)

info = topic_model.get_topic_info()
print(f"{(info['Topic'] != -1).sum()} topics found; "
      f"{(info.loc[info['Topic']==-1,'Count'].sum()):,} emails are outliers (-1)")
info.head(30)

## 7. INSPECT — do not skip this, this is the whole job
The keywords are statistical, not a label. Read the representative emails per topic and
ask: *is this a requirement, or an artifact (a template, boilerplate I failed to strip)?*
With **synthetic** emails this matters double — clusters may track the generator's
templates, not organic needs. Name the real ones yourself.

In [ ]:
for tid in info[info["Topic"] != -1]["Topic"].head(15):
    kw = ", ".join(w for w, _ in topic_model.get_topic(tid)[:8])
    print(f"\n=== TOPIC {tid}  (n={info.loc[info['Topic']==tid,'Count'].values[0]}) ===")
    print("keywords:", kw)
    for d in topic_model.get_representative_docs(tid)[:3]:
        print("  -", d[:200].replace("\n", " "))

## 8. (optional) Reassign outliers
Forces ambiguous emails into their nearest topic — convenient for coverage, but it
*manufactures* certainty the data did not have. Keep the `was_outlier` flag so downstream
can tell forced assignments apart. Leave False until you trust the topics.

In [ ]:
REASSIGN_OUTLIERS = False
if REASSIGN_OUTLIERS:
    new_topics = topic_model.reduce_outliers(docs, topics, strategy="c-tf-idf")
    pdf["topic"] = new_topics
else:
    pdf["topic"] = topics
pdf["was_outlier"] = [t == -1 for t in topics]
pdf["topic_prob"] = [float(p.max()) if hasattr(p, "max") else float(p) for p in probs]

## 9. Write to the SILVER lakehouse
The original `saveAsTable("silver.…")` does **not** reach a different lakehouse on Fabric
— the default catalog here is bronze. So we write Delta straight to the silver lakehouse's
`Tables/` path by its OneLake abfss URI (unambiguous, independent of which lakehouse is
attached). Fabric auto-registers Delta folders under `Tables/` as tables.

`get_topic_info()` returns list-typed columns (`Representation`, `Representative_Docs`)
that break `spark.createDataFrame`, so the summary is rebuilt from scalar columns only.

If `lh_silver_banking_data` is **schema-enabled**, change the paths to
`{SILVER_TABLES}/dbo/{ASSIGN_TABLE}` etc.

In [ ]:
# scalar-only frames (avoid list columns that break createDataFrame)
label_map = {
    tid: ", ".join(w for w, _ in topic_model.get_topic(tid)[:6]) if tid != -1 else "outlier"
    for tid in info["Topic"]
}
pdf["topic_keywords"] = pdf["topic"].map(label_map)

assign_df = pdf[[ID_COL, "topic", "topic_keywords", "topic_prob", "was_outlier"]].copy()
assign_df["topic"]       = assign_df["topic"].astype("int64")
assign_df["topic_prob"]  = assign_df["topic_prob"].astype("float64")
assign_df["was_outlier"] = assign_df["was_outlier"].astype(bool)

summary_df = info[["Topic", "Count", "Name"]].rename(
    columns={"Topic": "topic", "Count": "email_count", "Name": "bertopic_name"}
).copy()
summary_df["topic_keywords"] = summary_df["topic"].map(label_map)
summary_df["topic"]       = summary_df["topic"].astype("int64")
summary_df["email_count"] = summary_df["email_count"].astype("int64")

(spark.createDataFrame(assign_df)
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .save(f"{SILVER_TABLES}/{ASSIGN_TABLE}"))
(spark.createDataFrame(summary_df)
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .save(f"{SILVER_TABLES}/{SUMMARY_TABLE}"))

print(f"Written to silver: {ASSIGN_TABLE} ({len(assign_df):,} rows), "
      f"{SUMMARY_TABLE} ({len(summary_df):,} rows)")

## 10. Validation checklist — how not to fool yourself
- **Coherence read:** for each topic, do the 3 representative emails share an intent? If
  not, the cluster is an artifact — raise `MIN_TOPIC_SIZE` or improve cleaning.
- **Boilerplate leak:** is any top topic dominated by signature/disclaimer text? -> add
  the real pattern to `DISCLAIMER_PATTERNS`, re-run from cell 3.
- **Synthetic-template leak:** are topics tracking the *generator's* phrasing rather than
  a business need? On synthetic data this is the likeliest failure.
- **Outlier rate:** if >40-50% land in -1, the corpus is too heterogeneous for these
  settings; tune `MIN_TOPIC_SIZE`, don't just reassign.
- **Stability:** re-run with a different `RANDOM_STATE`. Themes that survive are real;
  themes that vanish were noise.